In [ ]:
import ray
import anthropic
import time
from tqdm.auto import tqdm
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns;sns.set_theme()
from wordcloud import WordCloud,STOPWORDS
import nltk
from nltk.corpus import stopwords
import json
# from nltk.stem import PorterStemmer
import re
import numpy as np
from transformers import BertTokenizer
from sklearn.metrics import precision_recall_fscore_support,classification_report,confusion_matrix,ConfusionMatrixDisplay
import sys
import os
import logging
logging.getLogger("httpx").setLevel(logging.WARNING)

TEST_SIZE=0.2
RANDOM_SEED=1234

In [ ]:
if ray.is_initialized():
    ray.shutdown()

In [ ]:
DATASET_LOC = "https://raw.githubusercontent.com/GokuMohandas/Made-With-ML/main/datasets/dataset.csv"
df = pd.read_csv(DATASET_LOC)
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.tag.value_counts()

In [ ]:
train_df, val_df = train_test_split(df,stratify=df.tag,test_size=TEST_SIZE,random_state=RANDOM_SEED)

In [ ]:
train_df.shape

In [ ]:
train_df.tag.value_counts()

In [ ]:
val_df.shape

In [ ]:
#verify if startify did its job
val_df.tag.value_counts() * int((1-TEST_SIZE)/TEST_SIZE)

In [ ]:
val_df.shape

In [ ]:
all_tags=Counter(df['tag'])
all_tags.most_common()
#Equivalent if no counter to be used
# count=df['tags'].value_counts()
# tag,tags_counts=count.index,count.values

In [ ]:
plt.figure(figsize=(10,5))
sns.countplot(df,x='tag',hue='tag',legend=False,palette='Set2',order=df['tag'].value_counts().index)
plt.tight_layout()

In [ ]:
# Plot tag frequencies #other alternative
tags, tag_counts=zip(*all_tags.most_common())
plt.figure(figsize=(10,3))
ax=sns.barplot(x=list(tags),y=list(tag_counts),hue=tags)
ax.set_xticklabels(tags,rotation=0)
plt.title('Tag Distribution',fontsize=14)
plt.ylabel('# of projects',fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
#most frequent tokens for each tag
for tag in tags: 
    plt.figure(figsize=(10,3))
    subset=df[df['tag']==tag]
    text=subset.title.values
    cloud=WordCloud(stopwords=STOPWORDS,background_color='black',collocations=False,width=500,height=300).generate(" ".join(text))
    plt.axis('off')
    print(tag)
    plt.imshow(cloud)

In [ ]:
df['text']=df['title']+ " " + df['description']

In [ ]:
#lru cache can be used as well 

def clean_text(text):  
    if not hasattr(clean_text,"_pattern"):
        nltk.download('stopwords',quiet=True)
        sw=stopwords.words('english')
        clean_text._pattern =re.compile(r'\b(' + r'|'.join(sw) + r")\b\s*")
    text=text.lower()    
    text=clean_text._pattern.sub('',text)
    text = re.sub(r"([!\"'#$%&()*\+,-./:;<=>?@\\\[\]^_`{|}~])", r" \1 ", text)  # add spacing before and after punctuations
    text = re.sub("[^A-Za-z0-9]+", " ", text)  # remove non alphanumeric chars , puncutations
    text = re.sub(" +", " ", text)  # remove multiple spaces
    text = text.strip()  # strip white space at the ends
    text = re.sub(r"http\S+", "", text)  #  remove links

    return text


In [ ]:
original_df=df.copy()
df['text']=df['text'].apply(clean_text)
print (f"{original_df.text.values[0]}\n{df.text.values[0]}")


In [ ]:
df=df.drop(columns=['id','created_on','title','description'],errors='ignore')
df=df.dropna(subset=['tag'])
df=df[['text','tag']]
df.head()

In [ ]:
#label encoding
tags=sorted(train_df['tag'].unique().tolist())
num_classes=len(tags)
class_to_index={tag:i for i, tag in enumerate(tags)}
class_to_index

In [ ]:
df['tag']=df['tag'].map(class_to_index)
df.head()

In [ ]:
def decode(indices,index_to_class):
    return [index_to_class[index] for index in indices]


In [ ]:
index_to_class={v:k for k,v in class_to_index.items()}
decode(df.head(1)['tag'].values,index_to_class=index_to_class)

In [ ]:
#bert tokenizer test
tokenizer=BertTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")
text="Transfer learning with transformers for text classification"
encoded_inputs=tokenizer([text],return_tensors='np',padding='longest')
print ("input_ids:", encoded_inputs["input_ids"])
print ("attention_mask:", encoded_inputs["attention_mask"])
print (tokenizer.decode(encoded_inputs["input_ids"][0]))



In [ ]:
#method to tokenize
def tokenize(batch):
    tokenizer=BertTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")
    encoded_inputs=tokenizer(batch['text'].tolist(),return_tensors='np',padding='longest',
                             truncation=True,max_length=128)
    return dict(ids=encoded_inputs["input_ids"],masks=encoded_inputs['attention_mask'],targets=np.array(batch['tag']))

In [ ]:
tokenize(df.head(2))

In [ ]:
# here tokenize is the issue , it gets reloaded every batch , changing this function to class will load once per worker 
def preprocess(df,class_to_index):
    df['text']=df['title']+" "+df['description'] #feature eng
    df['text']=df['text'].apply(clean_text) #clean_text
    df = df.drop(columns=["id", "created_on", "title", "description"], errors="ignore")  # clean dataframe
    df = df[["text", "tag"]]  # rearrange columns
    df["tag"] = df["tag"].map(class_to_index)  # label encoding
    outputs=tokenize(df)
    return outputs


In [ ]:
#just for refrence here. not used 
class Preprocessor:
    def __init__(self, class_to_index):
        self.tokenizer = BertTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")  # once per actor
        self.class_to_index = class_to_index


    def tokenize(self,df):
        encoded_inputs=self.tokenizer(df['text'].tolist(),return_tensors='np',padding='longest',
                             truncation=True,max_length=128)
        return dict(ids=encoded_inputs["input_ids"],masks=encoded_inputs['attention_mask'],targets=np.array(df['tag']))

    def __call__(self, df):
        # same body as your preprocess(), but use self.tokenizer inside tokenize
        df['text']=df['title']+" "+df['description'] #feature eng
        df['text']=df['text'].apply(clean_text) #clean_text
        df = df.drop(columns=["id", "created_on", "title", "description"], errors="ignore")  # clean dataframe
        df = df[["text", "tag"]]  # rearrange columns
        df["tag"] = df["tag"].map(class_to_index)  # label encoding    
        return self.tokenize(df)




In [ ]:
preprocess(df=train_df,class_to_index=class_to_index)

In [ ]:
ray.data.DatasetContext.get_current().execution_options.preserve_order=True #deterministic
ray.init()

In [ ]:
ray.cluster_resources()

In [ ]:
ds=ray.data.read_csv(DATASET_LOC,override_num_blocks=int(ray.cluster_resources().get('CPU',1)))
ds=ds.random_shuffle(seed=RANDOM_SEED)
ds.take(1)

In [ ]:
#ray dataset
def stratify_split(ds,stratify,test_size,shuffle=True,seed = 1234):

    def _add_split(df):
        train, test = train_test_split(df,test_size=test_size, shuffle=shuffle, random_state=seed)
        train["_split"] = "train"
        test["_split"] = "test"
        return pd.concat([train, test])

    def _filter_split(df, split):
        return df[df["_split"] == split].drop(columns="_split",errors="ignore")

    
    grouped=ds.groupby(stratify).map_groups(_add_split,batch_format='pandas')
    train_ds=grouped.map_batches(_filter_split,fn_kwargs={"split": "train"}, batch_format="pandas")
    test_ds=grouped.map_batches(_filter_split,fn_kwargs={"split": "test"}, batch_format="pandas")

    train_ds=train_ds.random_shuffle(seed=seed)
    test_ds=test_ds.random_shuffle(seed=seed)

    return train_ds,test_ds

In [ ]:
train_ds,val_ds=stratify_split(ds,stratify='tag',test_size=TEST_SIZE)
# train_ds

In [ ]:
#just to check  how to do in ray , earlier class to index can be used here 
#  tags=train_ds.unique(column='tag')
# class_to_index = {tag: i for i, tag in enumerate(tags)}
tags = sorted(train_ds.to_pandas()['tag'].unique()) #to pandas because tag column is small , ray overhead doesnt make sense.
class_to_index = {tag: i for i, tag in enumerate(tags)}

In [ ]:
# sample_ds_classOne = train_ds.map_batches(
#   Preprocessor,
#   fn_constructor_kwargs={"class_to_index": class_to_index},
#   batch_format="pandas")
# sample_ds.show(1)

sample_ds = train_ds.map_batches(
  preprocess,
  fn_kwargs={"class_to_index": class_to_index},
  batch_format="pandas")
sample_ds.show(1)

Lets establish baselines with zeroshot and few shot

In [ ]:
client =anthropic.Anthropic()


In [ ]:
# response= client.messages.create(
#     model='claude-sonnet-5',
#     max_tokens=100,
#     messages=[{'role': "user",'content':'tell me a joke'}]
# )
# [response.content[0].text][0]

In [ ]:
DATASET_LOC = "https://raw.githubusercontent.com/GokuMohandas/Made-With-ML/main/datasets/dataset.csv"
train_df = pd.read_csv(DATASET_LOC)
print(train_df.head())
tags=train_df.tag.unique().tolist()
print(tags)

In [ ]:
HOLDOUT_LOC = "https://raw.githubusercontent.com/GokuMohandas/Made-With-ML/main/datasets/holdout.csv"
test_df = pd.read_csv(HOLDOUT_LOC)

In [ ]:
test_df[['title','tag']][:3]

In [ ]:
# function that can predict tags for a given sample
def get_tag(model,system_content="",assistant_content="",user_content=""):
    messages=[]
    if assistant_content:
        messages.append({"role": "assistant", "content": assistant_content})
    messages.append({"role": "user", "content": user_content})
    try:
        response= client.messages.create(
        model=model,
        max_tokens=100,
        system=system_content,
        messages=messages,
        # temperature=0,        
        )
        text_blocks = [block.text for block in response.content if block.type == "text"]
        predicted_tag=text_blocks[0].strip().lower() if text_blocks else None
        return predicted_tag
    except (anthropic.APIError) as e:
        return None
        

In [ ]:
# List of dicts] as serving endpoint needs json format 
samples=test_df[["title","description"]].to_dict(orient='records')[:3]
samples

In [ ]:
#sanity
model='claude-sonnet-5'
system_context = f"""
    You are a NLP prediction service that predicts the label given an input's title and description.
    You must choose between one of the following labels for each input: {tags}.
    Only respond with the label name and nothing else.
    """
assistant_content=""
user_content ={'title': 'Diffusion to Vector',
  'description': 'Reference implementation of Diffusion2Vec (Complenet 2018) built on Gensim and NetworkX. '}
tag=get_tag(model=model,system_content=system_context,assistant_content=assistant_content,
            user_content=str(user_content)) #model expects user input in string format 
print(tag)




In [ ]:
def get_predictions(inputs,model,system_content,assistant_content=""):
    y_pred=[]
    for item in tqdm(inputs):
        user_content=str(item)
        # print(item)
        predicted_tag=get_tag(model,system_content,assistant_content,
                              user_content)
        while predicted_tag is None:
            time.sleep(30)
            predicted_tag=predicted_tag=get_tag(model,system_content,assistant_content,
                                          user_content)
        y_pred.append(predicted_tag)
    return y_pred


In [ ]:
get_predictions(samples,model,system_context)

In [ ]:
def clean_predictions(y_pred,tags,default='other'):
    for i, item in enumerate(y_pred):
        if item not in tags: #hallucination
            y_pred[i]=default
        if item.startswith("'") and item.endswith("'"): #destringify
            y_pred[i]=item[1:-1]
    return y_pred


In [ ]:
#plot ground truth and predictions
def plot_tag_dist(y_true,y_pred):
    true_tag_freq=dict(Counter(y_true))
    pred_tag_freq=dict(Counter(y_pred))
    df_true=pd.DataFrame({'tag':list(true_tag_freq.keys()),'freq':list(true_tag_freq.values()),'source': 'true'})
    df_pred=pd.DataFrame({'tag':list(pred_tag_freq.keys()),'freq':list(pred_tag_freq.values()),'source': 'pred'})
    df=pd.concat([df_true,df_pred],ignore_index=True)
    
    plt.figure(figsize=(10, 3))
    plt.title("Tag distribution", fontsize=14)
    ax = sns.barplot(x="tag", y="freq", hue="source", data=df)
    ax.set_xticklabels(list(true_tag_freq.keys()), rotation=0, fontsize=8)
    plt.legend()
    plt.show()



In [ ]:
def evaluate(test_df,model,system_content,tags,assistant_content=''):

    y_test=test_df['tag'].to_list()
    test_samples=test_df[['title','description']].to_dict(orient='records')
    y_pred=get_predictions(
        inputs=test_samples,model=model,
        system_content=system_content,assistant_content=assistant_content
    )
    y_pred=clean_predictions(y_pred=y_pred,tags=tags)


    metrics=precision_recall_fscore_support(y_test,y_pred,average='weighted')
    performance={'precision':metrics[0],'recall':metrics[1],'f1':metrics[2]}
    print(json.dumps(performance,indent=2))
    plot_tag_dist(y_true=y_test,y_pred=y_pred)
    return y_pred,performance

In [ ]:
#initialise dicts for analysis accros multiple models
y_pred = {"zero_shot": {}, "few_shot": {}}
performance = {"zero_shot": {}, "few_shot": {}}

In [ ]:
system_content = f"""
    You are a NLP prediction service that predicts the label given an input's title and description.
    You must choose between one of the following labels for each input: {tags}.
    Only respond with the label name and nothing else.
    """

In [ ]:
method='zero_shot'
model='claude-sonnet-5'
y_pred[method][model],performance[method][model]=evaluate(
    test_df=test_df,model=model,system_content=system_content,tags=tags
)

In [ ]:
#results seems to good , verifying with classification report 
y_test=test_df['tag'].to_list()
print(classification_report(y_test,y_pred[method][model], target_names=tags))

In [ ]:
#zeroshot on haiku
method='zero_shot'
model='claude-haiku-4-5-20251001'
y_pred[method][model],performance[method][model]=evaluate(
    test_df=test_df,model=model,system_content=system_content,tags=tags
)

In [ ]:
#pulling few shot examples from test set would be leakage
num_samples=2
additional_context=[]
cols_to_keep=["title","description","tag"]
for tag in tags:
    samples=train_df[cols_to_keep][train_df.tag==tag][:num_samples].to_dict(orient='records')
    additional_context.extend(samples)
additional_context

In [ ]:
# Add assistant context
assistant_content = f"""Here are some examples with the correct labels: {additional_context}"""
print (assistant_content)

In [ ]:
# Few-shot on sonnet 5
method = "few_shot"
model = "claude-sonnet-5"
y_pred[method][model], performance[method][model] = evaluate(
    test_df=test_df, model=model, system_content=system_content,
    assistant_content=assistant_content, tags=tags)

In [ ]:
# Few-shot on haiku
method = "few_shot"
model = "claude-haiku-4-5-20251001"
y_pred[method][model], performance[method][model] = evaluate(
    test_df=test_df, model=model, system_content=system_content,
    assistant_content=assistant_content, tags=tags)

In [ ]:
print(classification_report(y_true=y_test,y_pred=y_pred[method][model],target_names=tags))

In [ ]:
cm=confusion_matrix(y_test,y_pred[method][model],labels=tags)
ConfusionMatrixDisplay(cm,display_labels=tags).plot(xticks_rotation=45,colorbar=False,cmap='coolwarm')

In [ ]:
print(json.dumps(performance,indent=2))

In [ ]:
by_model_context={}
for context_type, models_data in performance.items():
    for model,metrics in models_data.items():
        key=f'{model}_{context_type}'
        by_model_context[key]=metrics

by_model_context

In [ ]:
# Plotting the bar chart with metric scores on top of each bar
models=list(by_model_context.keys())
metrics=list(by_model_context[models[0]].keys())

fig, ax = plt.subplots(figsize=(10, 4))
width = 0.2
x = range(len(models))

for i, metric in enumerate(metrics):
    metric_values = [by_model_context[model][metric] for model in models]
    ax.bar([pos + width * i for pos in x], metric_values, width, label=metric)
    # Displaying the metric scores on top of each bar
    for pos, val in zip(x, metric_values):
        ax.text(pos + width * i, val, f'{val:.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks([pos + width for pos in x])
ax.set_xticklabels(models, rotation=0, ha='center', fontsize=8)
ax.set_ylabel('Performance')
ax.set_title('Anthropic Benchmarks')
ax.legend(loc='upper left', bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.show()

In [ ]:
#another way as data is already in wider format , using pandas
ax = pd.DataFrame(by_model_context).T.plot.bar(figsize=(10, 4), rot=0, width=0.7,
                                               title="Anthropic Benchmarks", ylabel="Performance") #outer key becomes column , inner becomes row. so .T
for c in ax.containers:
    ax.bar_label(c, fmt="%.3f", fontsize=9)
ax.tick_params(axis="x", labelsize=8)
ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()


In [ ]:
#if was in longer formal then , example
df = (pd.DataFrame(by_model_context).T         
        .rename_axis("model").reset_index()
        .melt(id_vars="model", var_name="metric", value_name="score"))

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=df, x="model", y="score", hue="metric", errorbar=None, ax=ax)

for c in ax.containers:                          # one container per metric
    ax.bar_label(c, fmt="%.3f", fontsize=9)

ax.set(title="Anthropic Benchmarks", xlabel="", ylabel="Performance")
ax.tick_params(axis="x", labelsize=8)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()


In [ ]:
import random
import torch
from ray.data.preprocessor import Preprocessor

In [ ]:
def set_seeds(seed=42):
    """Set seeds for reproducibility."""
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    eval("setattr(torch.backends.cudnn, 'deterministic', True)")
    eval("setattr(torch.backends.cudnn, 'benchmark', False)")
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds()

In [ ]:
def load_data(num_samples=None):
    ds=ray.data.read_csv(DATASET_LOC,override_num_blocks=int(ray.cluster_resources().get('CPU',1)))
    ds=ds.random_shuffle(seed=RANDOM_SEED)
    ds=ray.data.from_items(ds.take(num_samples)) if num_samples else ds #truncate ray dataset if numsamples valid
    return ds


In [ ]:
# The leading underscores (_fit, _transform_pandas) are the hooks we override,always call the public fit / transform / fit_transform, which do bookkeeping and then call underscore methods
class CustomPreprocessor(Preprocessor):
    def _fit(self, ds):
        tags = sorted(ds.to_pandas()["tag"].unique())
        self.class_to_index = {tag: i for i, tag in enumerate(tags)}
        self.index_to_class = {v: k for k, v in self.class_to_index.items()}
        return self

    def _transform_pandas(self, batch):
        return preprocess(batch, class_to_index=self.class_to_index)

In [ ]:
import torch.nn as nn
from transformers import BertModel

In [ ]:
#PreTrained LLM
llm=BertModel.from_pretrained("allenai/scibert_scivocab_uncased",return_dict=False) #return dict for controlling output 
embedding_dim=llm.config.hidden_size

In [ ]:
llm

In [ ]:
print(list(llm.state_dict().keys()))


In [ ]:
# for name , p in llm.named_parameters():
#     print(f'{name} requires grad {p.requires_grad} {tuple(p.shape)}')

total=sum(p.numel() for p in llm.parameters())
learnt=sum(p.numel() for p in llm.parameters() if p.requires_grad)
learnable=learnt/total
print(f'total {total} learnable {100*learnable}')

for n ,_ in llm.named_buffers():
    print(n)

In [ ]:
x=llm.named_parameters()
list(x)

In [ ]:
y=list(llm.parameters())
y

In [ ]:
llm.state_dict()


In [ ]:
# Sample sanity 
text = "Transfer learning with transformers for text classification."
batch = tokenizer([text], return_tensors="pt", padding="longest")
# batch = {k:torch.tensor(v) for k,v in batch.items()}  # convert to torch tensors
seq, pool = llm(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
np.shape(seq), np.shape(pool)

In [ ]:
import torch.nn.functional as F

In [ ]:
class FinetunedLLM(nn.Module):
    def __init__(self,llm,embedding_dim,num_classes,dropout_p):
        super().__init__()
        self.llm=llm
        self.embedding_dim=embedding_dim
        self.num_classes=num_classes
        self.dropout_p=dropout_p
        self.dropout=nn.Dropout(dropout_p)
        self.fc1=nn.Linear(embedding_dim,num_classes)

    def forward(self,batch):
        ids,mask=batch['ids'],batch['masks']
        _,pool=self.llm(input_ids=ids,attention_mask=mask)
        logits=self.dropout(pool)
        logits=self.fc1(logits)
        return logits
    
    @torch.inference_mode()
    def predict(self,batch):
        self.eval()
        logits=self(batch)
        y_pred=torch.argmax(logits,dim=1).cpu().numpy()
        return y_pred

    @torch.inference_mode()
    def predict_proba(self,batch):
        self.eval()
        logits=self(batch)
        y_probs=F.softmax(logits,dim=1).cpu().numpy()
        return y_probs

    def save(self,dp):
        with open(Path(dp,"args.json"),'w') as fp:
            content={
                'embedding_dim':self.embedding_dim,
                'num_classes':self.num_classes,
                'dropout_p':self.dropout_p
            }
            json.dump(content,fp=fp,indent=4,sort_keys=False)
        torch.save(self.state_dict(),os.path.join(dp,'model.pt'))

    @classmethod
    def load(cls,args_fp,dict_fp):
        with open(args_fp,'r') as fp:
            kwargs=json.load(fp=fp)
        llm=BertModel.from_pretrained("allenai/scibert_scivocab_uncased",return_dict=False)
        model=cls(llm=llm,**kwargs)
        model.load_state_dict(torch.load(dict_fp,map_location=torch.device('cpu')))
        return model


In [ ]:
#intitialize model
model=FinetunedLLM(llm=llm,dropout_p=.5,embedding_dim=embedding_dim,num_classes=num_classes)
print(list(model.named_parameters()))


In [ ]:
llm.encoder.layer[1].attention.self.key.weight

In [ ]:
from ray.train.torch import get_device as ray_get_device
from functools import lru_cache

#Ray's get_device() inside a train worker, plain torch device on the driver."""
@lru_cache(maxsize=None)
def get_device():    
    try:
        return ray_get_device()
    except RuntimeError:  # not in a Ray Train worker (e.g. notebook driver)
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def pad_array(arr, dtype=np.int32):
    max_len = max(len(row) for row in arr)
    padded_arr = np.zeros((arr.shape[0], max_len), dtype=dtype) # 2d array of 0 of row, max len
    for i, row in enumerate(arr):
        padded_arr[i][:len(row)] = row
    return padded_arr

In [ ]:
def collate_fn(batch):
    batch["ids"] = pad_array(batch["ids"])
    batch["masks"] = pad_array(batch["masks"])
    dtypes = {"ids": torch.int32, "masks": torch.int32, "targets": torch.int64} #targets int64 only as loss fn  expects int64
    return {key: torch.tensor(array, dtype=dtypes[key], device=get_device()) for key,array in batch.items()}

In [ ]:
# Sample batch| dataset > shard (1 per worker) > block > rows, batch is collection of rows 
sample_batch = sample_ds.take_batch(batch_size=128)
collate_fn(batch=sample_batch)

In [ ]:
# Ray 2.58: ray.air.Checkpoint / ray.air.session were removed.
# Everything now lives under ray.train (use train.report / train.get_context /
# train.get_dataset_shard / train.get_checkpoint in place of `session.*`).
import tempfile


import ray.train as train
from ray.train import Checkpoint, CheckpointConfig, DataConfig, RunConfig, ScalingConfig
from ray.train.torch import TorchTrainer
import torch.nn.functional as F
from torch.nn.parallel.distributed import DistributedDataParallel


In [ ]:
def train_step(ds, batch_size, model, num_classes, loss_fn, optimizer):
    
    model.train()
    loss = 0.0
    ds_generator = ds.iter_torch_batches(batch_size=batch_size, collate_fn=collate_fn) #lazy iterator
    for i, batch in enumerate(ds_generator):
        optimizer.zero_grad()  # reset gradients
        z = model(batch)  # forward pass
        targets =batch["targets"] # one-hot (for loss_fn if BCE , CE doesnt need One hot for single label multiclass)
        J = loss_fn(z, targets)  # define loss
        J.backward()  # backward pass
        optimizer.step()  # update weights
        loss += (J.detach().item() - loss) / (i + 1)  # cumulative loss
    return loss

In [ ]:
def eval_step(ds, batch_size, model, num_classes, loss_fn):
    model.eval()
    loss = 0.0
    y_trues, y_preds = [], []
    ds_generator = ds.iter_torch_batches(batch_size=batch_size, collate_fn=collate_fn)
    with torch.inference_mode():
        for i, batch in enumerate(ds_generator):
            z = model(batch)
            targets = batch["targets"]
            J = loss_fn(z, targets).item()
            loss += (J - loss) / (i + 1) #better compute wise  to keep running average, we could do append and then average at the end 
            y_trues.extend(batch["targets"].cpu().numpy())
            y_preds.extend(torch.argmax(z, dim=1).cpu().numpy())
    return loss, np.vstack(y_trues), np.vstack(y_preds)

                     trainer.fit()
               ┌──────────┴──────────┐
        Worker 0 (cuda:0)     Worker 1 (cuda:1)
        shard 0 of data       shard 1 of data
        same code ↓           same code ↓
        load BERT             load BERT
        prepare_model ◄──sync──► prepare_model
        epoch loop:           epoch loop:
          each step ◄──grad avg──► each step
          report    ◄──wait────►  report

Step	                           What happens                                	Why
Read config       	            Get hyperparameters	                   One function, many configs
set_seeds	                      Seed this process	                     Reproducible runs
get_dataset_shard 	            Get this worker's data                 	Workers don't overlap
from_pretrained + FinetunedLLM	SciBERT + random head     	            Start of fine-tuning
prepare_model	                  To GPU + DDP + sync weights	            All copies start equal and stay in sync
Adam + scheduler	               Update rule + lr decay	                Learn nd take smaller steps when stuck
batch_size // world_size	       Split the batch	                      Same mini batch for any worker count
train_step / eval_step	          Learn / measure	                       One epoch each
save + report	                    Save weights + log metrics	          Pick the best model later and load it for inference 

In [ ]:
# Training loop
def train_loop_per_worker(config):
    # Hyperparameters
    dropout_p = config["dropout_p"]
    lr = config["lr"]
    lr_factor = config["lr_factor"]
    lr_patience = config["lr_patience"]
    num_epochs = config["num_epochs"]
    batch_size = config["batch_size"]
    num_classes = config["num_classes"]

    # Get datasets
    set_seeds()
    train_ds = train.get_dataset_shard("train")
    val_ds = train.get_dataset_shard("val")

    # Model
    llm = BertModel.from_pretrained("allenai/scibert_scivocab_uncased", return_dict=False)
    model = FinetunedLLM(llm=llm, dropout_p=dropout_p, embedding_dim=llm.config.hidden_size, num_classes=num_classes)
    model = train.torch.prepare_model(model) #now the model is DDP wrapper . FinetunedLLM is model.module

    # Training components always after prepare model so that it holds the paramter on already on the GPU  
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=lr_factor, patience=lr_patience)

    # Training
    batch_size_per_worker = batch_size // train.get_context().get_world_size() 
    for epoch in range(num_epochs):
        # Step
        train_loss = train_step(train_ds, batch_size_per_worker, model, num_classes, loss_fn, optimizer)
        val_loss, _, _ = eval_step(val_ds, batch_size_per_worker, model, num_classes, loss_fn)
        scheduler.step(val_loss)

        # Checkpoint
        with tempfile.TemporaryDirectory() as dp:
            if isinstance(model, DistributedDataParallel):  # cpu
                model.module.save(dp=dp) #Finetuned LLm's save method 
            else:
                model.save(dp=dp)
            metrics = dict(epoch=epoch, lr=optimizer.param_groups[0]["lr"], train_loss=train_loss, val_loss=val_loss)
            checkpoint = Checkpoint.from_directory(dp)
            train.report(metrics, checkpoint=checkpoint)


In [ ]:
#if imbalanced
#  # Class weights
# batch_counts = []
# for batch in train_ds.iter_torch_batches(batch_size=256, collate_fn=collate_fn):
#     batch_counts.append(np.bincount(batch["targets"].cpu().numpy()))
# counts = [sum(count) for count in zip(*batch_counts)]
# class_weights = np.array([1.0/count for i, count in enumerate(counts)])
# class_weights_tensor = torch.Tensor(class_weights).to(get_device())

# # Training components
# loss_fn = nn.BCEWithLogitsLoss(weight=class_weights_tensor)

In [ ]:
# Train loop config
train_loop_config = {
    "dropout_p": 0.5,
    "lr": 1e-4,
    "lr_factor": 0.8,
    "lr_patience": 3,
    "num_epochs": 5,
    "batch_size": 32,  # was 256; BERT fwd+bwd on CPU peaks at ~4.9 GB RSS at 128/worker
    "num_classes": num_classes,
}

In [ ]:
# Workers -- this machine is CPU-only (torch.cuda.is_available() is False) with 4 physical
# cores and ~17 GB RAM, of which only ~5 GB is typically free while the kernel is alive.
#
# Measured peak RSS for one worker doing BERT-base fwd+bwd+Adam at seq_len 40:
#   batch/worker  16 -> 2.29 GB | 32 -> 2.65 GB | 64 -> 3.39 GB | 128 -> 4.93 GB
# The previous setting (num_workers=2, batch_size=256 => 128/worker) needed ~9.9 GB and
# died with "MemoryError: Unable to allocate internal buffer" in the Ray deserializer.
#
# One worker holding all 4 physical cores is also no slower here: 2 workers x 2 threads
# splits the same cores and adds DDP gradient sync on top.
num_workers = 2
resources_per_worker = {"CPU": 4, "GPU": 0}

In [ ]:
# Scaling config
scaling_config = ScalingConfig(
    num_workers=num_workers,
    use_gpu=bool(resources_per_worker["GPU"]),
    resources_per_worker=resources_per_worker,
)


In [ ]:
# Run config
checkpoint_config = CheckpointConfig(num_to_keep=1, checkpoint_score_attribute="val_loss", checkpoint_score_order="min") # remove num to keep if we want all checkpoint to save. checkpoint is configured per epoch
run_config = RunConfig(
    name="llm",
    checkpoint_config=checkpoint_config,
    storage_path=os.path.expanduser("~/ray_results"),  # `local_dir` was removed in Ray 2.x; must be an absolute path
)


In [ ]:
ds = load_data()
train_ds, val_ds = stratify_split(ds, stratify="tag", test_size=TEST_SIZE)

In [ ]:
# Preprocess
preprocessor = CustomPreprocessor()
train_ds =  preprocessor.fit_transform(train_ds)
val_ds = preprocessor.transform(val_ds)
train_ds = train_ds.materialize() # run the pipeline once and cache it; otherwise it re-executes every epoch
val_ds = val_ds.materialize()

In [ ]:
# Dataset config
from ray.data import ExecutionOptions

dataset_config = DataConfig(
    datasets_to_split=["train", "val"],
    execution_options=ExecutionOptions(preserve_order=True), #default ray handes out blocks in order they arrive, for reproducability 
)


In [ ]:
# The driver still holds two SciBERT copies from the exploration cells above (~0.9 GB).
# train_loop_per_worker loads its own model, so dropping them to save ram and prevent OOM 
import gc
del llm, model
gc.collect()
torch.cuda.empty_cache() #on gpu 


# Trainer
trainer = TorchTrainer(
    train_loop_per_worker=train_loop_per_worker,
    train_loop_config=train_loop_config,
    scaling_config=scaling_config,
    run_config=run_config,
    datasets={"train": train_ds, "val": val_ds},
    dataset_config=dataset_config
)

In [ ]:
results = trainer.fit()

In [ ]:
results.metrics_dataframe

In [ ]:
results.best_checkpoints